In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf

## Load dataset

In [2]:
df = pd.read_csv("../data/dataset.csv")
print(df.head())

   hw_id  question_id student_id ta_id ta_role question_type  has_mistakes  \
0      1     55741749         s1    t1     UTA       applied             0   
1      1     55741749         s2    t1     UTA       applied             1   
2      1     55741749         s3    t1     UTA       applied             0   
3      1     55741749         s4    t1     UTA       applied             1   
4      1     55741749         s5    t1     UTA       applied             1   

   treatment  feedback_provision  feedback_stage_time  feedback_time  \
0          1                   0                0.950            NaN   
1          1                   0               51.487         51.487   
2          1                   0                5.956            NaN   
3          1                   0               19.669         19.669   
4          0                   0                0.737          0.737   

   adjusted_feedback_time  feedback_length  feedback_rating  draft_usage  \
0                     

In [3]:
print(df['question_id'].nunique())

36


## Compute Counts

Counts for outcomes present in both control and treatment conditions

In [4]:
cols = [
    "feedback_provision",
    "feedback_length",
    "feedback_stage_time",
    "feedback_time",
    "adjusted_feedback_time",
    "feedback_rating"
]

for col in cols:
    # when an outcome is undefined, its value is NaN
    # ---> feedback length (char) is only defined when feedback was given
    # ---> feedback time (s) is only defined when feedback was given
    # ---> adjusted feedback time (s/char) is only defined when feedback was given
    # ---> feedback rating (1-5 Likert) is only defined when student rated the received feedback
    count = df.loc[df[col].notna(), "treatment"].value_counts().sort_index()
    print(f"{col}")
    print(f"control count: {count[0]}")
    print(f"treatment count: {count[1]}\n")

feedback_provision
control count: 1418
treatment count: 1410

feedback_length
control count: 1418
treatment count: 1410

feedback_stage_time
control count: 1234
treatment count: 1227

feedback_time
control count: 276
treatment count: 388

adjusted_feedback_time
control count: 276
treatment count: 388

feedback_rating
control count: 214
treatment count: 254



Counts for outcomes only present in treatment condition

In [5]:
cols = [
    "draft_usage",
    "draft_rating",
    "similarity_score",
    "grade_reconsideration"
]

for col in cols:
    # we constrain 'draft usage' to instances when feedback is given
    # ---> our interpretation is of how much of the draft was used in the final feedback
    # otherwise, when an outcome is undefined, its value is NaN
    # ---> similarity score (rouge-l f1) is only defined when feedback was given
    if col == "draft_usage":
        count = df.loc[df["feedback_provision"] == 1, col].notna().sum()
    else:
        count = df[col].notna().sum()
    print(f"{col}")
    print(f"count: {count}\n")

draft_usage
count: 458

draft_rating
count: 1410

similarity_score
count: 432

grade_reconsideration
count: 1410



Counts for subgroups

In [8]:
df["ta_role"].value_counts()

ta_role
GTA    1522
UTA    1306
Name: count, dtype: int64

In [9]:
df["question_type"].value_counts()

question_type
applied       1453
conceptual    1375
Name: count, dtype: int64

In [10]:
df["has_mistakes"].value_counts()

has_mistakes
0    2194
1     634
Name: count, dtype: int64

## Regression Results

### Feedback Provision

In [6]:
cols = [
    "feedback_provision",
    "treatment",
    "question_id",      
    "student_id"
]
df_cleaned = df[cols].dropna().copy()

feedback_provision_m = smf.ols(
    "feedback_provision ~ treatment + C(question_id)",
    data=df_cleaned
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_cleaned["student_id"]}
)

print(feedback_provision_m.summary())

                            OLS Regression Results                            
Dep. Variable:     feedback_provision   R-squared:                       0.543
Model:                            OLS   Adj. R-squared:                  0.538
Method:                 Least Squares   F-statistic:                     1521.
Date:                Wed, 23 Sep 2026   Prob (F-statistic):          7.39e-108
Time:                        10:52:51   Log-Likelihood:                -610.62
No. Observations:                2828   AIC:                             1295.
Df Residuals:                    2791   BIC:                             1515.
Df Model:                          36                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

#### Feedback Provision by HW

In [7]:
models = {}

cols = [
    "feedback_provision",
    "treatment",
    "question_id",
    "student_id"
]

for hw in range(1, 5):
    df_cleaned = (
        df.loc[df["hw_id"] == hw, cols]
        .dropna()
        .copy()
    )

    model = smf.ols(
        "feedback_provision ~ treatment + C(question_id)",
        data=df_cleaned
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": df_cleaned["student_id"]}
    )

    models[hw] = model

results = pd.DataFrame([
    {
        "HW": hw,
        "Treatment Effect": model.params["treatment"],
        "SE": model.bse["treatment"],
        "p-value": model.pvalues["treatment"],
        "N": int(model.nobs)
    }
    for hw, model in models.items()
])

print(results.round(4))

   HW  Treatment Effect      SE  p-value    N
0   1            0.1059  0.0284   0.0002  631
1   2            0.0821  0.0182   0.0000  707
2   3            0.1171  0.0203   0.0000  767
3   4            0.1260  0.0192   0.0000  723


### Feedback Length

In [8]:
cols = [
    "feedback_length",
    "treatment",
    "question_id",      
    "student_id"
]
df_cleaned = df[cols].dropna().copy()

feedback_length_m = smf.ols(
    "feedback_length ~ treatment + C(question_id)",
    data=df_cleaned
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_cleaned["student_id"]}
)

print(feedback_length_m.summary())

                            OLS Regression Results                            
Dep. Variable:        feedback_length   R-squared:                       0.351
Model:                            OLS   Adj. R-squared:                  0.343
Method:                 Least Squares   F-statistic:                     28.34
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           1.04e-34
Time:                        10:52:52   Log-Likelihood:                -16497.
No. Observations:                2828   AIC:                         3.307e+04
Df Residuals:                    2791   BIC:                         3.329e+04
Df Model:                          36                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

### Feedback Stage Time

In [9]:
cols = [
    "feedback_stage_time",
    "treatment",
    "question_id",      
    "student_id"
]
df_cleaned = df[cols].dropna().copy()

feedback_time_m = smf.ols(
    "feedback_stage_time ~ treatment + C(question_id)",
    data=df_cleaned
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_cleaned["student_id"]}
)

print(feedback_time_m.summary())

                             OLS Regression Results                            
Dep. Variable:     feedback_stage_time   R-squared:                       0.014
Model:                             OLS   Adj. R-squared:                  0.001
Method:                  Least Squares   F-statistic:                     12.49
Date:                 Wed, 23 Sep 2026   Prob (F-statistic):           9.15e-21
Time:                         10:52:52   Log-Likelihood:                -14867.
No. Observations:                 2461   AIC:                         2.980e+04
Df Residuals:                     2429   BIC:                         2.998e+04
Df Model:                           31                                         
Covariance Type:               cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept 

### Feedback Time

In [10]:
cols = [
    "feedback_time",
    "treatment",
    "question_id",      
    "student_id"
]
df_cleaned = df[cols].dropna().copy()

feedback_time_m = smf.ols(
    "feedback_time ~ treatment + C(question_id)",
    data=df_cleaned
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_cleaned["student_id"]}
)

print(feedback_time_m.summary())

                            OLS Regression Results                            
Dep. Variable:          feedback_time   R-squared:                       0.169
Model:                            OLS   Adj. R-squared:                  0.128
Method:                 Least Squares   F-statistic:                     1252.
Date:                Wed, 23 Sep 2026   Prob (F-statistic):          8.55e-103
Time:                        10:52:52   Log-Likelihood:                -4375.3
No. Observations:                 664   AIC:                             8815.
Df Residuals:                     632   BIC:                             8959.
Df Model:                          31                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

### Adjusted Feedback Time

In [11]:
cols = [
    "adjusted_feedback_time",
    "treatment",
    "question_id",      
    "student_id"
]
df_cleaned = df[cols].dropna().copy()

feedback_time_m = smf.ols(
    "adjusted_feedback_time ~ treatment + C(question_id)",
    data=df_cleaned
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_cleaned["student_id"]}
)

print(feedback_time_m.summary())

                              OLS Regression Results                              
Dep. Variable:     adjusted_feedback_time   R-squared:                       0.164
Model:                                OLS   Adj. R-squared:                  0.123
Method:                     Least Squares   F-statistic:                     4776.
Date:                    Wed, 23 Sep 2026   Prob (F-statistic):          4.84e-128
Time:                            10:52:53   Log-Likelihood:                -1848.2
No. Observations:                     664   AIC:                             3760.
Df Residuals:                         632   BIC:                             3904.
Df Model:                              31                                         
Covariance Type:                  cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

### Feedback Rating

In [12]:
cols = [
    "feedback_rating",
    "treatment",
    "question_id",      
    "student_id"
]
df_cleaned = df[cols].dropna().copy()

feedback_rating_m = smf.ols(
    "feedback_rating ~ treatment + C(question_id)",
    data=df_cleaned
).fit(
    cov_type="cluster",
    cov_kwds={"groups": df_cleaned["student_id"]}
)

print(feedback_rating_m.summary())

                            OLS Regression Results                            
Dep. Variable:        feedback_rating   R-squared:                       0.065
Model:                            OLS   Adj. R-squared:                 -0.013
Method:                 Least Squares   F-statistic:                     5.860
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           1.59e-09
Time:                        10:52:53   Log-Likelihood:                -557.01
No. Observations:                 468   AIC:                             1188.
Df Residuals:                     431   BIC:                             1342.
Df Model:                          36                                         
Covariance Type:              cluster                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
Intercept           

## Descriptive Stats

### Helper Functions

In [13]:
def summarize_series(x, confidence=0.95):
    x = x.dropna()
    n = len(x)

    if n == 0:
        return pd.Series({
            "mean": np.nan,
            "se": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "n": 0
        })

    mean = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)

    if n == 1:
        margin = np.nan
    else:
        alpha = 1 - confidence
        t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
        margin = t_crit * se

    return pd.Series({
        "mean": mean,
        "se": se,
        "ci_lower": mean - margin if n > 1 else np.nan,
        "ci_upper": mean + margin if n > 1 else np.nan,
        "n": n
    })

In [14]:
def summarize_outcomes(
    df,
    outcome_cols,
    group_cols=None,
    treatment_only=False,
    filters=None,
    confidence=0.95
):
    if group_cols is None:
        group_cols = []

    rows = []

    for outcome in outcome_cols:
        data = df.copy()

        # some outcomes are only defined for the treatment (e.g., draft rating)
        if treatment_only:
            data = data[data["treatment"] == 1]

        # draft usage is defined only when feedback is given
        # so we filter for cases when when feedback is given via lambda function
        if filters and outcome in filters:
            data = data[filters[outcome](data)]

        x = data[outcome]

        if outcome == "grade_reconsideration":
            data = data.copy()
            data[outcome] = data[outcome].str.lower().map({
                "keep": 0,
                "increase": 1,
                "decrease": 1
            })

        grouping = group_cols.copy()

        if not treatment_only:
            grouping = grouping + ["treatment"]

        if grouping:
            grouped = data.groupby(grouping, dropna=False)
            for group_values, group_df in grouped:
                if not isinstance(group_values, tuple):
                    group_values = (group_values,)

                summary = summarize_series(group_df[outcome], confidence)

                row = {"outcome": outcome}
                row.update(dict(zip(grouping, group_values)))
                row.update(summary.to_dict())
                rows.append(row)
        else:
            summary = summarize_series(data[outcome], confidence)
            row = {"outcome": outcome}
            row.update(summary.to_dict())
            rows.append(row)

    return pd.DataFrame(rows)

### Shared Outcomes

#### Course-Level

In [15]:
shared_results = summarize_outcomes(
    df,
    ["feedback_provision", "feedback_length", "feedback_stage_time", "feedback_time", "adjusted_feedback_time","feedback_rating"],
    treatment_only=False
)
shared_results.to_csv("../tables/shared_results.csv")
print(shared_results)

                   outcome  treatment       mean         se   ci_lower  \
0       feedback_provision          0   0.217207   0.010954   0.195719   
1       feedback_provision          1   0.324823   0.012476   0.300349   
2          feedback_length          0  25.098025   1.954936  21.263145   
3          feedback_length          1  65.041844   3.247555  58.671280   
4      feedback_stage_time          0   3.828657   0.702857   2.449729   
5      feedback_stage_time          1  14.355229   4.076979   6.356601   
6            feedback_time          0   8.339768   1.553724   5.281064   
7            feedback_time          1  32.292235  12.762016   7.200671   
8   adjusted_feedback_time          0   0.269426   0.048018   0.174896   
9   adjusted_feedback_time          1   0.451449   0.282517  -0.104012   
10         feedback_rating          0   3.467290   0.053254   3.362317   
11         feedback_rating          1   3.484252   0.054007   3.377891   

     ci_upper       n  
0    0.238695

#### Homework-Level

In [24]:
shared_by_hw = summarize_outcomes(
    df,
    ["feedback_provision", "feedback_length", "feedback_stage_time", "feedback_time", "adjusted_feedback_time", "feedback_rating"],
    group_cols=["hw_id"],
    treatment_only=False
)
shared_by_hw.to_csv("../tables/shared_by_hw.csv")
print(shared_by_hw)

                   outcome  hw_id  treatment        mean         se  \
0       feedback_provision      1          0    0.398089   0.027668   
1       feedback_provision      1          1    0.501577   0.028127   
2       feedback_provision      2          0    0.176471   0.020205   
3       feedback_provision      2          1    0.251429   0.023223   
4       feedback_provision      3          0    0.238961   0.021762   
5       feedback_provision      3          1    0.358639   0.024571   
6       feedback_provision      4          0    0.077348   0.014060   
7       feedback_provision      4          1    0.204986   0.021276   
8          feedback_length      1          0   53.296178   5.908182   
9          feedback_length      1          1  123.621451   8.560636   
10         feedback_length      2          0    8.834734   1.665198   
11         feedback_length      2          1   44.677143   4.801682   
12         feedback_length      3          0   34.400000   4.558512   
13    

#### Stratified 

In [26]:
strat_col = "has_mistakes"
shared_stratified = summarize_outcomes(
    df,
    ["feedback_provision", "feedback_length", "feedback_stage_time", "feedback_time", "adjusted_feedback_time","feedback_rating"],
    group_cols=[strat_col],
    treatment_only=False
)
shared_stratified.to_csv(f"../tables/shared_stratified_{strat_col}.csv")
print(shared_stratified)

                   outcome  has_mistakes  treatment        mean         se  \
0       feedback_provision             0          0    0.100358   0.008999   
1       feedback_provision             0          1    0.166976   0.011364   
2       feedback_provision             1          0    0.649007   0.027510   
3       feedback_provision             1          1    0.837349   0.020285   
4          feedback_length             0          0    2.742832   0.435261   
5          feedback_length             0          1   19.867347   1.868630   
6          feedback_length             1          0  107.708609   7.285276   
7          feedback_length             1          1  211.722892   8.292003   
8      feedback_stage_time             0          0    2.827827   0.767191   
9      feedback_stage_time             0          1    7.539755   0.733015   
10     feedback_stage_time             1          0    7.728718   1.686101   
11     feedback_stage_time             1          1   37.406136 

### Treatment-Only Outcomes

#### Course-Level

In [20]:
treatment_results = summarize_outcomes(
    df,
    ["draft_usage", "draft_rating", "similarity_score", "grade_reconsideration"],
    treatment_only=True,
    filters={
        "draft_usage": lambda d: d["feedback_provision"] == 1
    }
)
treatment_results.to_csv("../tables/treatment_results.csv")
print(treatment_results)

                 outcome      mean        se  ci_lower  ci_upper       n
0            draft_usage  0.541485  0.023308  0.495680  0.587290   458.0
1           draft_rating  3.002128  0.028600  2.946025  3.058230  1410.0
2       similarity_score  0.595937  0.016623  0.563265  0.628610   432.0
3  grade_reconsideration  0.039007  0.005158  0.028889  0.049125  1410.0


#### Homework-Level

In [21]:
treatment_by_hw = summarize_outcomes(
    df,
    ["draft_usage", "draft_rating", "similarity_score", "grade_reconsideration"],
    group_cols=["hw_id"],
    treatment_only=True,
    filters={
        "draft_usage": lambda d: d["feedback_provision"] == 1
    }
)
treatment_by_hw.to_csv("../tables/treatment_by_hw.csv")
print(treatment_by_hw)

                  outcome  hw_id      mean        se  ci_lower  ci_upper  \
0             draft_usage      1  0.597484  0.039015  0.520427  0.674542   
1             draft_usage      2  0.784091  0.044112  0.696413  0.871769   
2             draft_usage      3  0.430657  0.042460  0.346689  0.514625   
3             draft_usage      4  0.337838  0.055357  0.227511  0.448165   
4            draft_rating      1  3.384858  0.060367  3.266086  3.503631   
5            draft_rating      2  3.082857  0.050100  2.984321  3.181393   
6            draft_rating      3  2.835079  0.058060  2.720921  2.949236   
7            draft_rating      4  2.764543  0.054286  2.657786  2.871300   
8        similarity_score      1  0.605758  0.027279  0.551901  0.659615   
9        similarity_score      2  0.655951  0.033001  0.590368  0.721533   
10       similarity_score      3  0.532714  0.030881  0.471616  0.593813   
11       similarity_score      4  0.623223  0.051050  0.520338  0.726108   
12  grade_re

#### Stratified

In [22]:
strat_col = "ta_role"
treatment_stratified = summarize_outcomes(
    df,
    ["draft_usage", "draft_rating", "similarity_score", "grade_reconsideration"],
    group_cols=[strat_col],
    treatment_only=True,
    filters={
        "draft_usage": lambda d: d["feedback_provision"] == 1
    }
)
treatment_stratified.to_csv(f"../tables/treatment_stratified_{strat_col}.csv")
print(treatment_stratified)

                 outcome ta_role      mean        se  ci_lower  ci_upper  \
0            draft_usage     GTA  0.820000  0.031474  0.757807  0.882193   
1            draft_usage     UTA  0.405844  0.028026  0.350697  0.460991   
2           draft_rating     GTA  2.786755  0.039732  2.708756  2.864753   
3           draft_rating     UTA  3.250382  0.038982  3.173837  3.326926   
4       similarity_score     GTA  0.758832  0.025553  0.708342  0.809321   
5       similarity_score     UTA  0.508403  0.019689  0.469645  0.547161   
6  grade_reconsideration     GTA  0.035762  0.006763  0.022486  0.049037   
7  grade_reconsideration     UTA  0.042748  0.007910  0.027216  0.058280   

       n  
0  150.0  
1  308.0  
2  755.0  
3  655.0  
4  151.0  
5  281.0  
6  755.0  
7  655.0  
